# Ciber-Inmunología Swarm en Google Colab

Este notebook corre el proyecto completo en Colab usando su GPU. Los nodos son simulados, así que toda la federación ocurre acá; al final quedan en tu Google Drive los resultados (CSV y PNG) y los pesos de los modelos (`.pt`).

**Antes de empezar:** activá la GPU en el menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**.

No hace falta subir nada: el dataset CIC-IDS2017 se descarga solo a tu Drive la primera vez.

Ejecutá las celdas en orden.

## 1. Comprobar la GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Descargar el código

In [ ]:
import os

%cd /content
if not os.path.exists("ciber-inmunologia"):
    !git clone -q https://github.com/xeviercr4/ciber-inmunologia.git
%cd /content/ciber-inmunologia
!git pull -q

## 3. Conectar Google Drive y obtener el dataset

Colab va a pedir permiso para acceder a tu Drive. Ahí se crea la carpeta `ciber-inmunologia` con dos subcarpetas:

- `raw`: los 8 CSV de CIC-IDS2017. **La primera vez se descargan solos** (unos 885 MB, tarda unos minutos); en las sesiones siguientes se toman de Drive.
- `resultados`: lo que generan los experimentos. Se guarda directo en Drive, así no se pierde si la sesión se corta.

Los CSV se bajan de una copia pública en Hugging Face. La fuente oficial es el [Canadian Institute for Cybersecurity](https://www.unb.ca/cic/datasets/ids-2017.html), que pide completar un formulario; si preferís esa, subí vos los 8 CSV a `ciber-inmunologia/raw` y esta celda usará los tuyos.

In [ ]:
import shutil
from google.colab import drive

drive.mount("/content/drive")

CARPETA_DRIVE = "/content/drive/MyDrive/ciber-inmunologia"
os.makedirs(f"{CARPETA_DRIVE}/raw", exist_ok=True)
os.makedirs(f"{CARPETA_DRIVE}/resultados", exist_ok=True)
os.makedirs("data/raw", exist_ok=True)

# Los experimentos leen esta variable y guardan sus resultados en Drive
os.environ["CIBER_RESULTADOS"] = f"{CARPETA_DRIVE}/resultados"

ORIGEN = "https://huggingface.co/datasets/c01dsnap/CIC-IDS2017/resolve/main"
DIAS = [
    "Monday-WorkingHours",
    "Tuesday-WorkingHours",
    "Wednesday-workingHours",
    "Thursday-WorkingHours-Morning-WebAttacks",
    "Thursday-WorkingHours-Afternoon-Infilteration",
    "Friday-WorkingHours-Morning",
    "Friday-WorkingHours-Afternoon-PortScan",
    "Friday-WorkingHours-Afternoon-DDos",
]

for dia in DIAS:
    nombre = f"{dia}.pcap_ISCX.csv"
    en_drive = f"{CARPETA_DRIVE}/raw/{nombre}"
    en_colab = f"data/raw/{nombre}"
    if os.path.exists(en_colab):
        continue
    if os.path.exists(en_drive):
        print(f"Desde Drive:  {nombre}")
        shutil.copy(en_drive, en_colab)      # leer del disco de Colab es mucho más rápido que de Drive
    else:
        print(f"Descargando:  {nombre}")
        !wget -q -O "{en_colab}" "{ORIGEN}/{nombre}"
        shutil.copy(en_colab, en_drive)      # copia permanente en Drive

print()
!ls -lh data/raw

## 4. Instalar Flower y verificar el entorno

Al instalar, pip muestra en rojo una lista larga de "dependency conflicts" (protobuf, click, cryptography). **Es esperable y no es un error del proyecto:** Flower necesita versiones más viejas de esas librerías que otros paquetes preinstalados en Colab (TensorFlow, wandb, Google Cloud), que acá no se usan.

Lo importante es que al final diga `Dispositivo: cuda` y `Entorno verificado correctamente`.

In [ ]:
!pip install -q -r requirements-colab.txt
!python verificar_entorno.py

## 5. Preprocesar los datos

`MUESTRA` es la fracción del dataset que se usa. Empezá con `0.1` (10%) para medir tiempos y subila después.

In [ ]:
MUESTRA = 0.1
!python -m src.preprocess --sample $MUESTRA

## 6. Experimentos

Elegí el modelo (`mlp` es rápido; `transformer` es el modelo principal y tarda más) y la cantidad de rondas. Cada experimento guarda su CSV y su gráfica en Drive al terminar.

Ejecutá un experimento por vez: no se pueden correr dos simulaciones al mismo tiempo.

In [ ]:
MODELO = "transformer"
RONDAS = 20

In [ ]:
# E1 — centralizado vs. aislado vs. federado
!python -m experiments.e1_comparacion --modelo $MODELO --rondas $RONDAS

In [ ]:
# E2 — FedAvg vs. FedProx vs. cabezas locales, con alpha = 0.1, 0.5 y 1.0
!python -m experiments.e2_heterogeneidad --modelo $MODELO --rondas $RONDAS

In [ ]:
# E3 — zero-day simulado (se puede cambiar la familia: DoS, DDoS, PortScan, BruteForce, Web, Bot)
!python -m experiments.e3_zero_day --modelo $MODELO --rondas $RONDAS --familia DDoS

In [ ]:
# E4 — 1 a 3 nodos maliciosos: FedAvg vs. Krum vs. mediana
!python -m experiments.e4_maliciosos --modelo $MODELO --rondas $RONDAS

## 7. Ver las gráficas

In [ ]:
import glob
from IPython.display import Image, display

for ruta in sorted(glob.glob(f"{CARPETA_DRIVE}/resultados/*.png")):
    print(os.path.basename(ruta))
    display(Image(ruta, width=900))

## 8. Llevar los resultados a tu computadora

Todo quedó en tu Drive, en `ciber-inmunologia/resultados` (los pesos de los modelos están en la subcarpeta `modelos`). Esta celda además arma un zip y lo descarga.

Para evaluar en tu máquina un modelo entrenado acá, copiá el `.pt` y ejecutá:

```
python -m src.centralized --evaluar ruta/al/archivo.pt
```

Importante: tu máquina tiene que haber preprocesado los mismos datos con la misma muestra (`--sample`) que se usó acá, porque el modelo depende de las columnas y de la normalización.

In [ ]:
from google.colab import files

shutil.make_archive("/content/resultados", "zip", f"{CARPETA_DRIVE}/resultados")
files.download("/content/resultados.zip")